In [222]:
# CELL 1: IMPORTS AND CONFIGURATION

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)
import xgboost as xgb
import joblib
import os
import json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# 🔧 Determine project root and models directory (robust across execution dirs)
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd
for p in [cwd] + list(cwd.parents):
    if (p / 'README.md').exists() or (p / '.git').exists():
        PROJECT_ROOT = p
        break
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Models directory: {MODELS_DIR}")

print("=" * 70)
print(" " * 15 + "BITCOIN PRICE PREDICTION")
print(" " * 15 + "XGBOOST MODEL")
print("=" * 70)

# ✅ SAFE ROC-AUC FUNCTION
def safe_roc_auc(y_true, y_proba):
    """Compute ROC AUC safely; return NaN if only one class present."""
    try:
        if len(np.unique(y_true)) < 2:
            return float('nan')
        return roc_auc_score(y_true, y_proba)
    except Exception as e:
        print(f"⚠️ ROC-AUC error: {e}")
        return float('nan')

Models directory: C:\Users\15086\Documents\GitHub\DeepBTC\models
               BITCOIN PRICE PREDICTION
               XGBOOST MODEL


In [223]:
# CELL 2: LOAD DATASET

print("\n[1/10] Loading dataset...")

df = pd.read_csv('../data/features/btc_features_complete.csv', index_col=0, parse_dates=True)

print(f"✓ Dataset loaded successfully")
print(f"  - Total rows: {df.shape[0]:,}")
print(f"  - Total columns: {df.shape[1]}")
print(f"  - Date range: {df.index.min()} → {df.index.max()}")
print(f"  - Frequency: {pd.infer_freq(df.index) or 'Irregular'}")


[1/10] Loading dataset...
✓ Dataset loaded successfully
  - Total rows: 51,443
  - Total columns: 86
  - Date range: 2020-01-31 00:00:00 → 2025-12-14 18:00:00
  - Frequency: Irregular


In [224]:
# CELL 3: CREATE TARGET VARIABLE

print("\n[2/10] Creating target variable...")

y = (df['target_direction_1h'] > 0).astype(int)

class_distribution = y.value_counts().sort_index()
print(f"✓ Target variable created: 'price_direction_1h'")
print(f"  - Class 0 (DOWN): {class_distribution[0]:,} samples ({class_distribution[0]/len(y)*100:.1f}%)")
print(f"  - Class 1 (UP):   {class_distribution[1]:,} samples ({class_distribution[1]/len(y)*100:.1f}%)")

balance_ratio = min(class_distribution) / max(class_distribution)
print(f"  - Balance ratio: {balance_ratio:.2f} (1.0 = perfectly balanced)")


[2/10] Creating target variable...
✓ Target variable created: 'price_direction_1h'
  - Class 0 (DOWN): 25,320 samples (49.2%)
  - Class 1 (UP):   26,123 samples (50.8%)
  - Balance ratio: 0.97 (1.0 = perfectly balanced)


In [225]:
# CELL 4: PREPARE FEATURES

print("\n[3/10] Preparing features...")

# Drop target-related and non-predictive columns
drop_cols = [c for c in df.columns if 'target' in c.lower()] + [
    'Datetime', 'Close', 'future_return_1h', 'future_return_6h', 'future_return_24h'
]
X = df.drop(columns=drop_cols, errors='ignore')

# Encode fear_greed_classification if present
if 'fear_greed_classification' in X.columns:
    print("✓ Encoding 'fear_greed_classification' feature...")
    sentiment_mapping = {
        'Extreme Fear': 0, 
        'Fear': 1, 
        'Neutral': 2, 
        'Greed': 3, 
        'Extreme Greed': 4
    }
    X['fear_greed_classification_num'] = X['fear_greed_classification'].map(sentiment_mapping)
    X = X.drop(columns=['fear_greed_classification'])

# Keep only numeric features
X = X.select_dtypes(include=[np.number])

print(f"✓ Feature selection completed")
print(f"  - Number of features: {X.shape[1]}")
print(f"  - Sample features: {list(X.columns[:5])}")


[3/10] Preparing features...
✓ Encoding 'fear_greed_classification' feature...
✓ Feature selection completed
  - Number of features: 79
  - Sample features: ['Open', 'High', 'Low', 'Volume', 'returns']


In [226]:
# CELL 5: CHRONOLOGICAL SPLIT

print("\n[4/10] Splitting data (chronological split)...")

n = len(X)
test_size = int(n * 0.15)
val_size = int(n * 0.15)
train_size = n - test_size - val_size

X_train = X.iloc[:train_size].copy()
y_train = y.iloc[:train_size].copy()
X_val = X.iloc[train_size:train_size+val_size].copy()
y_val = y.iloc[train_size:train_size+val_size].copy()
X_test = X.iloc[train_size+val_size:].copy()
y_test = y.iloc[train_size+val_size:].copy()

print(f"✓ Data split completed (70% / 15% / 15%)")
print(f"\n  Training set: {X_train.shape[0]:,} samples | {X_train.index.min()} → {X_train.index.max()}")
print(f"  Validation set: {X_val.shape[0]:,} samples | {X_val.index.min()} → {X_val.index.max()}")
print(f"  Test set: {X_test.shape[0]:,} samples | {X_test.index.min()} → {X_test.index.max()}")


[4/10] Splitting data (chronological split)...
✓ Data split completed (70% / 15% / 15%)

  Training set: 36,011 samples | 2020-01-31 00:00:00 → 2024-03-11 18:00:00
  Validation set: 7,716 samples | 2024-03-11 19:00:00 → 2025-01-27 06:00:00
  Test set: 7,716 samples | 2025-01-27 07:00:00 → 2025-12-14 18:00:00


In [227]:
# CELL 6: CLEANING AND SCALING

print("\n[5/10] Cleaning and scaling data...")

# Replace infinite values with NaN
for df_part in [X_train, X_val, X_test]:
    df_part.replace([np.inf, -np.inf], np.nan, inplace=True)

nan_counts = X_train.isna().sum()
total_nans = nan_counts.sum()
features_with_nan = (nan_counts > 0).sum()

print(f"✓ Infinite values replaced with NaN")
print(f"  - Total NaN values: {total_nans:,}")
print(f"  - Features with NaN: {features_with_nan}/{X_train.shape[1]}")

# Compute and save medians from training set
medians = X_train.median()
joblib.dump(medians, str(MODELS_DIR / 'xgb_medians.pkl'))

# Fill missing values
X_train.fillna(medians, inplace=True)
X_val.fillna(medians, inplace=True)
X_test.fillna(medians, inplace=True)

print(f"✓ Missing values imputed using training set medians (saved to {MODELS_DIR / 'xgb_medians.pkl'})")

# Note: XGBoost can handle non-scaled data, but scaling can help
# We'll scale for consistency with logistic regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f"✓ Features standardized (StandardScaler)")


[5/10] Cleaning and scaling data...
✓ Infinite values replaced with NaN
  - Total NaN values: 0
  - Features with NaN: 0/79
✓ Missing values imputed using training set medians (saved to C:\Users\15086\Documents\GitHub\DeepBTC\models\xgb_medians.pkl)
✓ Features standardized (StandardScaler)


In [228]:
# CELL 7: TRAIN OPTIMIZED XGBOOST MODEL (Version équilibrée - Moins de sous-apprentissage que la précédente tentative)
# Cette version ajuste les hyperparamètres pour réduire le surapprentissage de la version de base
# tout en évitant le sous-apprentissage excessif de la version "Balanced Complexity".
# L'objectif est d'améliorer potentiellement l'accuracy sur le test set.

import time
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

print("\n[6/10] Training Optimized XGBoost model (Refined Balance)...\n")

start_time = time.time()

# ========================================================================
# 1. GESTION DU DÉSÉQUILIBRE DES CLASSES
# ========================================================================
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"  📊 Class Distribution:")
print(f"    • Class 0 (DOWN): {len(y_train[y_train == 0]):,}")
print(f"    • Class 1 (UP):   {len(y_train[y_train == 1]):,}")
print(f"    • Scale Pos Weight: {scale_pos_weight:.4f}")

# ========================================================================
# 2. PARAMÈTRES XGBOOST OPTIMISÉS (Réduction modérée du surapprentissage)
# ========================================================================
# Cette configuration est un compromis entre la version de base et la version très régularisée.
# Elle vise à réduire le surapprentissage sans trop pénaliser la capacité d'apprentissage.
xgb_params = {
    # Paramètres de base
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'logloss', 'error'],  # Suivi de l'accuracy pendant l'entraînement
    'tree_method': 'hist',  # Méthode d'entraînement plus rapide
    'random_state': 42,
    'n_jobs': -1, # Utilise tous les cœurs disponibles

    # Complexité de l'arbre (légèrement réduite)
    'max_depth': 4,        # Conservé de la version de base
    'min_child_weight': 6, # Augmenté de 5 à 6 (un peu plus conservateur que la base, moins que la version échouée)
    'max_delta_step': 1,   # Aide pour les classes déséquilibrées

    # Paramètres d'apprentissage
    'learning_rate': 0.028, # Légèrement réduit de 0.03 (un peu plus stable que la base)
    'n_estimators': 550,   # Augmenté de 500, mais avec early stopping

    # Régularisation (légèrement augmentée par rapport à la base)
    'gamma': 0.25,          # Légèrement augmenté de 0.2
    'reg_alpha': 0.4,       # L1 augmentée de 0.3 (moins que la version échouée)
    'reg_lambda': 2.2,      # L2 augmentée de 2.0 (moins que la version échouée)

    # Échantillonnage (légèrement réduit par rapport à la base)
    'subsample': 0.68,      # Réduit de 0.7
    'colsample_bytree': 0.68, # Réduit de 0.7
    'colsample_bylevel': 0.68, # Ajouté: échantillonnage des colonnes par niveau
    'colsample_bynode': 0.68,  # Ajouté: échantillonnage des colonnes par noeud

    # Déséquilibre des classes
    'scale_pos_weight': scale_pos_weight,

    # Early stopping (toujours actif)
    'early_stopping_rounds': 35, # Un peu plus tolérant que la base (30), moins que la version échouée (40)
    'verbose': 0 # Désactive les logs pendant l'entraînement
}

print(f"\n  🔧 Optimized Configuration (Refined Balance):")
print(f"    • Max Depth: {xgb_params['max_depth']}")
print(f"    • Min Child Weight: {xgb_params['min_child_weight']} (↑ from 5)")
print(f"    • Learning Rate: {xgb_params['learning_rate']} (↓ from 0.03)")
print(f"    • N Estimators: {xgb_params['n_estimators']}")
print(f"    • Regularization: γ={xgb_params['gamma']}, α={xgb_params['reg_alpha']}, λ={xgb_params['reg_lambda']}")
print(f"    • Sampling Rates: {xgb_params['subsample']}, {xgb_params['colsample_bytree']} (↓ from 0.7)")
print(f"    • Early Stopping Rounds: {xgb_params['early_stopping_rounds']}")

# ========================================================================
# 3. ENTRAÎNEMENT AVEC JEU D'ÉVALUATION MULTIPLE
# ========================================================================
print(f"\n  🚀 Training in progress...")

xgb_model = xgb.XGBClassifier(**xgb_params)

# Entraînement avec jeu d'évaluation
xgb_model.fit(
    X_train_s,
    y_train,
    eval_set=[
        (X_train_s, y_train),  # Suivi de l'entraînement
        (X_val_s, y_val)       # Suivi de la validation
    ],
    verbose=False
)

training_time = time.time() - start_time

# ========================================================================
# 4. RÉSULTATS D'ENTRAÎNEMENT DÉTAILLÉS
# ========================================================================
print(f"\n  ✓ Model trained successfully!")
print(f"    • Training time: {training_time:.2f}s")
print(f"    • Best iteration (based on validation AUC): {xgb_model.best_iteration}")
print(f"    • Best validation AUC score: {xgb_model.best_score:.4f}")

# Récupération des métriques d'entraînement
train_results = xgb_model.evals_result()
try:
    train_error = train_results['validation_0']['error'][xgb_model.best_iteration]
    val_error = train_results['validation_1']['error'][xgb_model.best_iteration]
    train_auc = train_results['validation_0']['auc'][xgb_model.best_iteration]
    val_auc = train_results['validation_1']['auc'][xgb_model.best_iteration]
    train_accuracy = 1 - train_error
    val_accuracy_at_best_auc = 1 - val_error # Accuracy au meilleur point AUC
    print(f"    • Training AUC: {train_auc:.4f}")
    print(f"    • Validation AUC: {val_auc:.4f}")
    print(f"    • Training Accuracy: {train_accuracy:.4f}")
    print(f"    • Validation Accuracy (at best AUC): {val_accuracy_at_best_auc:.4f}")
    print(f"    • Overfitting gap (AUC): {(train_auc - val_auc):.4f}")
    print(f"    • Overfitting gap (Accuracy): {(train_accuracy - val_accuracy_at_best_auc):.4f}")
except (KeyError, IndexError):
    print("    ⚠️ Could not retrieve training accuracy/error from evals_result.")

# Vérification du surapprentissage (toujours basé sur AUC principalement)
if (train_auc - val_auc) > 0.05:
    print(f"    ⚠️  WARNING: Significant overfitting (AUC gap) still detected!")
    print(f"       Consider increasing regularization further or reducing complexity.")
elif (train_auc - val_auc) < 0.02:
    print(f"    ✅ Good generalization (low AUC overfitting).")
else:
    print(f"    ⚡ Moderate AUC overfitting (acceptable).")

# ========================================================================
# 5. DIAGNOSTICS DU MODÈLE
# ========================================================================
print(f"\n  🔬 Model Diagnostics:")
print(f"    • Number of trees used (at best AUC): {xgb_model.best_iteration + 1}")
epochs = len(train_results['validation_0'].get('auc', []))
print(f"    • Total training rounds attempted: {epochs}")
print(f"    • Early stopping triggered: {'Yes' if epochs > xgb_model.best_iteration + xgb_params['early_stopping_rounds'] else 'No'}")

if xgb_model.best_iteration < 50:
    print(f"    ⚠️  Model converged very early - may be underfitting.")
    print(f"       Consider: increasing learning_rate or reducing regularization.")
elif xgb_model.best_iteration >= epochs:
    print(f"    ⚠️  Model hit maximum rounds ({xgb_params['n_estimators']}) - might need more rounds or less regularization.")
else:
    print(f"    ✅ Convergence looks healthy.")

print(f"\n{'='*70}")


[6/10] Training Optimized XGBoost model (Refined Balance)...

  📊 Class Distribution:
    • Class 0 (DOWN): 17,697
    • Class 1 (UP):   18,314
    • Scale Pos Weight: 0.9663

  🔧 Optimized Configuration (Refined Balance):
    • Max Depth: 4
    • Min Child Weight: 6 (↑ from 5)
    • Learning Rate: 0.028 (↓ from 0.03)
    • N Estimators: 550
    • Regularization: γ=0.25, α=0.4, λ=2.2
    • Sampling Rates: 0.68, 0.68 (↓ from 0.7)
    • Early Stopping Rounds: 35

  🚀 Training in progress...

  ✓ Model trained successfully!
    • Training time: 1.39s
    • Best iteration (based on validation AUC): 44
    • Best validation AUC score: 0.4680
    • Training AUC: 0.5921
    • Validation AUC: 0.5447
    • Training Accuracy: 0.5647
    • Validation Accuracy (at best AUC): 0.5320
    • Overfitting gap (AUC): 0.0474
    • Overfitting gap (Accuracy): 0.0327
    ⚡ Moderate AUC overfitting (acceptable).

  🔬 Model Diagnostics:
    • Number of trees used (at best AUC): 45
    • Total training rounds

In [229]:
# CELL 8: EVALUATE ON VALIDATION SET

print("\n[7/10] Evaluating model on validation set...")

y_val_pred = xgb_model.predict(X_val_s)
y_val_proba = xgb_model.predict_proba(X_val_s)[:, 1]

# Calculate metrics
val_accuracy = accuracy_score(y_val, y_val_pred)
val_precision = precision_score(y_val, y_val_pred, zero_division=0)
val_recall = recall_score(y_val, y_val_pred, zero_division=0)
val_roc_auc = safe_roc_auc(y_val, y_val_proba)
cm_val = confusion_matrix(y_val, y_val_pred)

print("\n" + "=" * 70)
print(" " * 20 + "VALIDATION SET RESULTS")
print("=" * 70)
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, digits=4))

print("Performance Metrics:")
print(f"  • Accuracy:  {val_accuracy:.4f}")
print(f"  • Precision: {val_precision:.4f} (of predicted UPs, how many were correct)")
print(f"  • Recall:    {val_recall:.4f} (of actual UPs, how many we caught)")
if not np.isnan(val_roc_auc):
    print(f"  • ROC-AUC:   {val_roc_auc:.4f}")
else:
    print(f"  • ROC-AUC:   N/A")

print("\nConfusion Matrix:")
print(f"                    Predicted")
print(f"                    DOWN      UP")
print(f"  Actual DOWN     {cm_val[0,0]:6d}   {cm_val[0,1]:6d}")
print(f"  Actual UP       {cm_val[1,0]:6d}   {cm_val[1,1]:6d}")

# Performance assessment
if np.isnan(val_roc_auc):
    performance = "⚠️ N/A - Check data"
elif val_roc_auc < 0.55:
    performance = "⚠️ POOR - Barely better than random"
elif val_roc_auc < 0.65:
    performance = "⚡ FAIR - Moderate predictive power"
elif val_roc_auc < 0.75:
    performance = "✓ GOOD - Decent predictive ability"
else:
    performance = "🌟 EXCELLENT - Strong predictive power"

print(f"\nOverall Performance: {performance}")


[7/10] Evaluating model on validation set...

                    VALIDATION SET RESULTS

Classification Report:
              precision    recall  f1-score   support

           0     0.5221    0.5427    0.5322      3785
           1     0.5423    0.5218    0.5318      3931

    accuracy                         0.5320      7716
   macro avg     0.5322    0.5322    0.5320      7716
weighted avg     0.5324    0.5320    0.5320      7716

Performance Metrics:
  • Accuracy:  0.5320
  • Precision: 0.5423 (of predicted UPs, how many were correct)
  • Recall:    0.5218 (of actual UPs, how many we caught)
  • ROC-AUC:   0.5447

Confusion Matrix:
                    Predicted
                    DOWN      UP
  Actual DOWN       2054     1731
  Actual UP         1880     2051

Overall Performance: ⚠️ POOR - Barely better than random


In [230]:
# CELL 9: BACKTEST FUNCTION

def realistic_backtest(y_true, y_pred, prices, initial_capital=10000, transaction_fee=0.001):
    """
    Simulate a trading strategy based on predictions
    
    Parameters:
    -----------
    y_true : array-like
        True labels (not used in strategy, only for evaluation)
    y_pred : array-like
        Predicted labels (1=UP/BUY, 0=DOWN/SELL)
    prices : pandas Series
        Bitcoin prices with datetime index
    initial_capital : float
        Starting capital in USD
    transaction_fee : float
        Transaction fee as decimal (0.001 = 0.1%)
    
    Returns:
    --------
    dict : Dictionary containing backtest results
    """
    cash = initial_capital
    btc_held = 0
    portfolio_values = []
    trades = []
    num_trades = 0
    
    for i in range(len(y_pred)):
        current_price = prices.iloc[i]
        timestamp = prices.index[i]
        
        # BUY signal (prediction UP)
        if y_pred[i] == 1 and cash > 0:
            btc_to_buy = cash / current_price
            btc_fee = btc_to_buy * transaction_fee
            btc_held = btc_to_buy - btc_fee
            usd_fee = btc_fee * current_price
            trades.append({
                'timestamp': timestamp,
                'action': 'BUY',
                'price': current_price,
                'btc_amount': btc_held,
                'fee': usd_fee
            })
            cash = 0
            num_trades += 1
            
        # SELL signal (prediction DOWN)
        elif y_pred[i] == 0 and btc_held > 0:
            cash_from_sale = btc_held * current_price
            usd_fee = cash_from_sale * transaction_fee
            trades.append({
                'timestamp': timestamp,
                'action': 'SELL',
                'price': current_price,
                'btc_amount': btc_held,
                'fee': usd_fee
            })
            cash = cash_from_sale - usd_fee
            btc_held = 0
            num_trades += 1
        
        # Calculate total portfolio value
        total_value = cash + (btc_held * current_price)
        portfolio_values.append(total_value)
    
    # Final metrics
    final_price = prices.iloc[-1]
    final_value = cash + (btc_held * final_price)
    total_return = (final_value - initial_capital) / initial_capital * 100
    
    # Buy and hold benchmark
    btc_bought_hold = initial_capital / prices.iloc[0]
    buy_hold_value = btc_bought_hold * final_price
    buy_hold_return = (buy_hold_value - initial_capital) / initial_capital * 100
    
    # Calculate Sharpe ratio
    returns = pd.Series(portfolio_values).pct_change().dropna()
    sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(252*24) if returns.std() > 0 else 0
    
    # Calculate max drawdown
    cumulative = pd.Series(portfolio_values)
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    
    return {
        'initial_capital': initial_capital,
        'final_value': final_value,
        'total_return': total_return,
        'buy_hold_value': buy_hold_value,
        'buy_hold_return': buy_hold_return,
        'outperformance': total_return - buy_hold_return,
        'portfolio_values': portfolio_values,
        'num_trades': num_trades,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'trades': trades
    }

print("✓ Backtest function defined")

✓ Backtest function defined


In [231]:
# CELL 10: VALIDATION BACKTEST WITH OPTIMIZATION

print("\n[8/10] Running OPTIMIZED backtest on validation set...")

val_prices = df.loc[X_val.index, 'Close']

# ✅ TEST MULTIPLE CONFIGURATIONS
configs = [
    {'name': 'Baseline (Original)', 'min_confidence': 0.0, 'cooldown_hours': 0, 'stop_loss_pct': 0, 'take_profit_pct': 0},
    {'name': 'Conservative', 'min_confidence': 0.65, 'cooldown_hours': 6, 'stop_loss_pct': 0.03, 'take_profit_pct': 0.05},
    {'name': 'Moderate', 'min_confidence': 0.60, 'cooldown_hours': 3, 'stop_loss_pct': 0.03, 'take_profit_pct': 0.05},
    {'name': 'Aggressive', 'min_confidence': 0.55, 'cooldown_hours': 1, 'stop_loss_pct': 0.02, 'take_profit_pct': 0.04}
]

results = []

for config in configs:
    bt = optimized_backtest(
        y_val, 
        y_val_pred, 
        y_val_proba,  # ✅ Pass probabilities
        val_prices, 
        initial_capital=10000, 
        transaction_fee=0.001,
        min_confidence=config['min_confidence'],
        cooldown_hours=config['cooldown_hours'],
        stop_loss_pct=config['stop_loss_pct'],
        take_profit_pct=config['take_profit_pct']
    )
    results.append((config['name'], bt))

# ========================================================================
# DISPLAY COMPARISON
# ========================================================================
print("\n" + "=" * 90)
print(" " * 30 + "STRATEGY COMPARISON")
print("=" * 90)
print(f"\n{'Strategy':<20} {'Return':>10} {'Trades':>8} {'Sharpe':>8} {'Win Rate':>10} {'vs B&H':>10}")
print("-" * 90)

for name, bt in results:
    print(f"{name:<20} {bt['total_return']:>9.2f}% {bt['num_trades']:>8} {bt['sharpe_ratio']:>8.2f} {bt['win_rate']*100:>9.1f}% {bt['outperformance']:>9.2f}%")

print("-" * 90)
print(f"{'Buy & Hold':<20} {results[0][1]['buy_hold_return']:>9.2f}%")
print("=" * 90)

# Find best strategy
best_strategy = max(results, key=lambda x: x[1]['total_return'])
print(f"\n🏆 BEST STRATEGY: {best_strategy[0]}")
print(f"   Return: {best_strategy[1]['total_return']:.2f}%")
print(f"   Trades: {best_strategy[1]['num_trades']}")
print(f"   Filter Rate: {best_strategy[1]['filter_rate']:.1f}%")

# Display sample trades from best strategy
val_backtest = best_strategy[1]  # Use best for later comparison

if val_backtest['num_trades'] > 0:
    print(f"\n📈 Sample Trades (first 5):")
    for trade in val_backtest['trades'][:5]:
        action = trade['action']
        conf = trade.get('confidence', 0)
        pl = trade.get('profit_loss', 0)
        print(f"  • {trade['timestamp']} | {action:12s} | {trade['btc_amount']:.6f} BTC @ ${trade['price']:,.2f} | "
              f"Conf: {conf:.2%} | P/L: {pl:+.2f}% | Fee: ${trade['fee']:.2f}")


[8/10] Running OPTIMIZED backtest on validation set...

                              STRATEGY COMPARISON

Strategy                 Return   Trades   Sharpe   Win Rate     vs B&H
------------------------------------------------------------------------------------------
Baseline (Original)    -100.00%     2719   -63.55       0.0%   -137.50%
Conservative              0.00%        0     0.00       0.0%    -37.50%
Moderate                -41.96%        2    -0.92       0.0%    -79.46%
Aggressive             -100.00%     1051   -27.37      15.1%   -137.50%
------------------------------------------------------------------------------------------
Buy & Hold               37.50%

🏆 BEST STRATEGY: Conservative
   Return: 0.00%
   Trades: 0
   Filter Rate: 100.0%


In [232]:
# CELL 11: FINAL EVALUATION ON TEST SET

print("\n[9/10] Final evaluation on test set...")

y_test_pred = xgb_model.predict(X_test_s)
y_test_proba = xgb_model.predict_proba(X_test_s)[:, 1]

# Calculate test metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_roc_auc = safe_roc_auc(y_test, y_test_proba)
cm_test = confusion_matrix(y_test, y_test_pred)

print("\n" + "=" * 70)
print(" " * 22 + "TEST SET RESULTS")
print("=" * 70)
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, digits=4))

print("Performance Metrics:")
print(f"  • Accuracy:  {test_accuracy:.4f}")
print(f"  • Precision: {test_precision:.4f}")
print(f"  • Recall:    {test_recall:.4f}")
if not np.isnan(test_roc_auc):
    print(f"  • ROC-AUC:   {test_roc_auc:.4f}")
else:
    print(f"  • ROC-AUC:   N/A")

print("\nConfusion Matrix:")
print(f"                    Predicted")
print(f"                    DOWN      UP")
print(f"  Actual DOWN     {cm_test[0,0]:6d}   {cm_test[0,1]:6d}")
print(f"  Actual UP       {cm_test[1,0]:6d}   {cm_test[1,1]:6d}")

# Test set backtest
test_prices = df.loc[X_test.index, 'Close']
test_backtest = realistic_backtest(
    y_test, 
    y_test_pred, 
    test_prices, 
    initial_capital=10000, 
    transaction_fee=0.001
)

print("\n" + "=" * 70)
print(" " * 22 + "TEST BACKTEST RESULTS")
print("=" * 70)
print(f"\n  Starting Capital:       ${test_backtest['initial_capital']:,.2f}")
print(f"  Final Portfolio Value:  ${test_backtest['final_value']:,.2f}")
print(f"  Total Return:           {test_backtest['total_return']:+.2f}%")
print(f"  Number of Trades:       {test_backtest['num_trades']}")
print(f"  Sharpe Ratio:           {test_backtest['sharpe_ratio']:.2f}")
print(f"  Max Drawdown:           {test_backtest['max_drawdown']:.2f}%")
print(f"  Buy & Hold Return:      {test_backtest['buy_hold_return']:+.2f}%")
print(f"  Outperformance:         {test_backtest['outperformance']:+.2f}%")


[9/10] Final evaluation on test set...

                      TEST SET RESULTS

Classification Report:
              precision    recall  f1-score   support

           0     0.5241    0.5211    0.5226      3838
           1     0.5287    0.5317    0.5302      3878

    accuracy                         0.5264      7716
   macro avg     0.5264    0.5264    0.5264      7716
weighted avg     0.5264    0.5264    0.5264      7716

Performance Metrics:
  • Accuracy:  0.5264
  • Precision: 0.5287
  • Recall:    0.5317
  • ROC-AUC:   0.5368

Confusion Matrix:
                    Predicted
                    DOWN      UP
  Actual DOWN       2000     1838
  Actual UP         1816     2062

                      TEST BACKTEST RESULTS

  Starting Capital:       $10,000.00
  Final Portfolio Value:  $1,199.93
  Total Return:           -88.00%
  Number of Trades:       2109
  Sharpe Ratio:           -6.38
  Max Drawdown:           -88.53%
  Buy & Hold Return:      -10.15%
  Outperformance:         

In [233]:
# CELL 12: FEATURE IMPORTANCE ANALYSIS

print("\n" + "=" * 70)
print(" " * 20 + "FEATURE IMPORTANCE")
print("=" * 70)

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print("-" * 70)
for idx, row in feature_importance.head(20).iterrows():
    print(f"  {row['feature']:40s} {row['importance']:.6f}")

print("\n" + "=" * 70)

# Save feature importance
feature_importance.to_csv(str(MODELS_DIR / 'xgb_feature_importance.csv'), index=False)
print(f"✓ Feature importance saved to {MODELS_DIR / 'xgb_feature_importance.csv'}")


                    FEATURE IMPORTANCE

Top 20 Most Important Features:
----------------------------------------------------------------------
  WILLR_14                                 0.034514
  returns                                  0.030175
  STOCHh_14_3_3                            0.028431
  log_returns                              0.027099
  BBP_20_2.0_2.0                           0.024111
  price_to_sma20                           0.017828
  RSI_21                                   0.017631
  STOCHd_14_3_3                            0.017325
  RSI_14                                   0.017007
  STOCHk_14_3_3                            0.016765
  price_momentum_24h                       0.015691
  DMP_14                                   0.015675
  EMA_12                                   0.015471
  nvt_ratio                                0.015115
  SMA_200                                  0.014967
  Volume                                   0.014772
  price_momentum_7d     

In [234]:
# CELL 13: SAVE MODEL AND ARTIFACTS

print("\n[10/10] Saving model and artifacts...")

# Save model
joblib.dump(xgb_model, str(MODELS_DIR / 'xgboost_model.pkl'))
joblib.dump(scaler, str(MODELS_DIR / 'xgb_scaler.pkl'))

# Save feature names
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, str(MODELS_DIR / 'xgb_feature_names.pkl'))

# Create metrics dictionary
metrics = {
    'model_type': 'XGBoost Classifier',
    'training_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_params': xgb_params,
    'data_info': {
        'total_samples': len(df),
        'num_features': X_train.shape[1],
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'class_balance': {
            'class_0': int(class_distribution[0]),
            'class_1': int(class_distribution[1]),
            'ratio': float(balance_ratio)
        }
    },
    'validation': {
        'accuracy': float(val_accuracy),
        'precision': float(val_precision),
        'recall': float(val_recall),
        'roc_auc': float(val_roc_auc) if not np.isnan(val_roc_auc) else None,
        'backtest_return': float(val_backtest['total_return']),
        'buy_hold_return': float(val_backtest['buy_hold_return']),
        'outperformance': float(val_backtest['outperformance']),
        'num_trades': int(val_backtest['num_trades']),
        'sharpe_ratio': float(val_backtest['sharpe_ratio']),
        'max_drawdown': float(val_backtest['max_drawdown'])
    },
    'test': {
        'accuracy': float(test_accuracy),
        'precision': float(test_precision),
        'recall': float(test_recall),
        'roc_auc': float(test_roc_auc) if not np.isnan(test_roc_auc) else None,
        'backtest_return': float(test_backtest['total_return']),
        'buy_hold_return': float(test_backtest['buy_hold_return']),
        'outperformance': float(test_backtest['outperformance']),
        'num_trades': int(test_backtest['num_trades']),
        'sharpe_ratio': float(test_backtest['sharpe_ratio']),
        'max_drawdown': float(test_backtest['max_drawdown'])
    },
    'best_iteration': int(xgb_model.best_iteration),
    'best_score': float(xgb_model.best_score)
}

# Save metrics
with open(str(MODELS_DIR / 'xgb_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"✓ Model saved:           {MODELS_DIR / 'xgboost_model.pkl'}")
print(f"✓ Scaler saved:          {MODELS_DIR / 'xgb_scaler.pkl'}")
print(f"✓ Feature names saved:   {MODELS_DIR / 'xgb_feature_names.pkl'}")
print(f"✓ Medians saved:         {MODELS_DIR / 'xgb_medians.pkl'}")
print(f"✓ Metrics saved:         {MODELS_DIR / 'xgb_metrics.json'}")

print("\n" + "=" * 70)
print(" " * 20 + "✅ TRAINING COMPLETED!")
print("=" * 70)


[10/10] Saving model and artifacts...
✓ Model saved:           C:\Users\15086\Documents\GitHub\DeepBTC\models\xgboost_model.pkl
✓ Scaler saved:          C:\Users\15086\Documents\GitHub\DeepBTC\models\xgb_scaler.pkl
✓ Feature names saved:   C:\Users\15086\Documents\GitHub\DeepBTC\models\xgb_feature_names.pkl
✓ Medians saved:         C:\Users\15086\Documents\GitHub\DeepBTC\models\xgb_medians.pkl
✓ Metrics saved:         C:\Users\15086\Documents\GitHub\DeepBTC\models\xgb_metrics.json

                    ✅ TRAINING COMPLETED!


In [235]:
# CELL 14: PREDICTION UTILITY FUNCTION

def predict_for_date(date_str, model, scaler, dataframe):
    """
    Make predictions for a specific date
    
    Parameters:
    -----------
    date_str : str
        Date string (e.g., '2024-12-12')
    model : XGBoost model
        Trained XGBoost model
    scaler : StandardScaler
        Fitted scaler
    dataframe : pandas DataFrame
        Complete dataset with features
    
    Returns:
    --------
    pandas DataFrame : Predictions with timestamps, direction, and probabilities
    """
    target_date = pd.to_datetime(date_str)
    
    # Prepare features (same as training)
    drop_cols = [c for c in dataframe.columns if 'target' in c.lower()] + [
        'Datetime', 'Close', 'future_return_1h', 'future_return_6h', 'future_return_24h'
    ]
    X = dataframe.drop(columns=drop_cols, errors='ignore')
    
    # Encode fear_greed_classification
    if 'fear_greed_classification' in X.columns:
        mapping = {
            'Extreme Fear': 0, 
            'Fear': 1, 
            'Neutral': 2, 
            'Greed': 3, 
            'Extreme Greed': 4
        }
        X['fear_greed_classification_num'] = X['fear_greed_classification'].map(mapping)
        X = X.drop(columns=['fear_greed_classification'])
    
    X = X.select_dtypes(include=[np.number])
    
    # Filter for target date
    X_pred = X.loc[dataframe.index.date == target_date.date()]
    
    if X_pred.empty:
        print(f"❌ No data available for {date_str}")
        return None
    
    # Clean data
    X_pred = X_pred.replace([np.inf, -np.inf], np.nan)
    
    # Load saved medians and feature names
    try:
        medians = joblib.load(str(MODELS_DIR / 'xgb_medians.pkl'))
    except Exception:
        medians = None
    
    try:
        feature_names = joblib.load(str(MODELS_DIR / 'xgb_feature_names.pkl'))
    except Exception:
        feature_names = None
    
    # Align features
    if feature_names is not None:
        for col in feature_names:
            if col not in X_pred.columns:
                fill_val = medians[col] if (medians is not None and col in medians.index) else 0
                X_pred[col] = fill_val
        X_pred = X_pred[feature_names]
    
    # Fill missing values
    if medians is not None:
        X_pred = X_pred.fillna(medians)
    else:
        X_pred = X_pred.fillna(X_pred.median())
    
    # Scale and predict
    X_pred_s = scaler.transform(X_pred)
    predictions = model.predict(X_pred_s)
    probabilities = model.predict_proba(X_pred_s)[:, 1]
    
    # Create results dataframe
    results = pd.DataFrame({
        'timestamp': X_pred.index,
        'prediction': ['UP' if p == 1 else 'DOWN' for p in predictions],
        'probability_up': probabilities,
        'confidence': [prob if pred == 1 else 1-prob for pred, prob in zip(predictions, probabilities)]
    })
    
    return results

print("✓ Prediction function defined")

✓ Prediction function defined


In [236]:
# CELL 15: TEST ON SPECIFIC DATE

def run_test_for_date(date_str, model, scaler, dataframe, transaction_fee=0.001):
    """
    Test predictions and backtest for a specific date
    
    Parameters:
    -----------
    date_str : str
        Date to test (e.g., '2024-12-12')
    model : trained model
        XGBoost model
    scaler : fitted scaler
        StandardScaler
    dataframe : pandas DataFrame
        Complete dataset
    transaction_fee : float
        Transaction fee as decimal
    
    Returns:
    --------
    dict : Test results including metrics and backtest
    """
    print("\n" + "=" * 70)
    print(f" TEST FOR DATE: {date_str} ")
    print("=" * 70)
    
    # Get predictions
    preds = predict_for_date(date_str, model, scaler, dataframe)
    if preds is None:
        return None
    
    # Get true labels
    mask = dataframe.index.date == pd.to_datetime(date_str).date()
    y_true = (dataframe.loc[mask, 'target_direction_1h'] > 0).astype(int)
    
    # Align predictions with true labels
    preds = preds.set_index('timestamp')
    preds = preds.loc[preds.index.isin(y_true.index)]
    y_true = y_true.loc[preds.index]
    
    if len(preds) == 0:
        print("❌ No overlapping timestamps")
        return None
    
    # Convert predictions to binary
    y_pred_bin = (preds['prediction'] == 'UP').astype(int).values
    y_proba = preds['probability_up'].values
    
    # Calculate metrics
    acc = accuracy_score(y_true, y_pred_bin)
    prec = precision_score(y_true, y_pred_bin, zero_division=0)
    rec = recall_score(y_true, y_pred_bin, zero_division=0)
    auc = safe_roc_auc(y_true, y_proba)
    
    print(f"\n  📊 Classification Metrics:")
    print(f"    • Samples:   {len(y_true)}")
    print(f"    • Accuracy:  {acc:.4f}")
    print(f"    • Precision: {prec:.4f}")
    print(f"    • Recall:    {rec:.4f}")
    if not np.isnan(auc):
        print(f"    • ROC-AUC:   {auc:.4f}")
    else:
        print(f"    • ROC-AUC:   N/A")
    
    # Run backtest
    prices = dataframe.loc[preds.index, 'Close']
    bt = realistic_backtest(
        y_true.values, 
        y_pred_bin, 
        prices, 
        initial_capital=10000, 
        transaction_fee=transaction_fee
    )
    
    print(f"\n  💰 Backtest Results:")
    print(f"    • Final Value:      ${bt['final_value']:,.2f}")
    print(f"    • Total Return:     {bt['total_return']:+.2f}%")
    print(f"    • Trades:           {bt['num_trades']}")
    print(f"    • Sharpe Ratio:     {bt['sharpe_ratio']:.2f}")
    print(f"    • Max Drawdown:     {bt['max_drawdown']:.2f}%")
    print(f"    • Buy & Hold:       {bt['buy_hold_return']:+.2f}%")
    print(f"    • Outperformance:   {bt['outperformance']:+.2f}%")
    
    # Display sample trades
    if bt['num_trades'] > 0 and len(bt['trades']) > 0:
        print(f"\n  📈 Sample Trades (first 5):")
        for trade in bt['trades'][:5]:
            print(f"    • {trade['timestamp']} | {trade['action']:4s} | "
                  f"{trade['btc_amount']:.6f} BTC @ ${trade['price']:,.2f} | "
                  f"Fee: ${trade['fee']:.2f}")
    
    return {
        'metrics': {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'roc_auc': auc
        },
        'backtest': bt,
        'predictions': preds
    }

# 🎯 RUN TEST FOR 2024-12-12
print("\n\n🔍 RUNNING TEST FOR DATE: 2024-12-12...\n")
test_result_2024_12_12 = run_test_for_date('2024-12-12', xgb_model, scaler, df)

print("\n" + "=" * 70)
print("✅ ALL OPERATIONS COMPLETED SUCCESSFULLY!")
print("=" * 70)



🔍 RUNNING TEST FOR DATE: 2024-12-12...


 TEST FOR DATE: 2024-12-12 

  📊 Classification Metrics:
    • Samples:   24
    • Accuracy:  0.4583
    • Precision: 0.4000
    • Recall:    0.6000
    • ROC-AUC:   0.6000

  💰 Backtest Results:
    • Final Value:      $9,969.63
    • Total Return:     -0.30%
    • Trades:           9
    • Sharpe Ratio:     -1.32
    • Max Drawdown:     -1.89%
    • Buy & Hold:       -0.76%
    • Outperformance:   +0.46%

  📈 Sample Trades (first 5):
    • 2024-12-12 00:00:00 | BUY  | 0.099134 BTC @ $100,772.46 | Fee: $10.00
    • 2024-12-12 02:00:00 | SELL | 0.099134 BTC @ $101,824.76 | Fee: $10.09
    • 2024-12-12 03:00:00 | BUY  | 0.099718 BTC @ $101,025.87 | Fee: $10.08
    • 2024-12-12 08:00:00 | SELL | 0.099718 BTC @ $100,980.00 | Fee: $10.07
    • 2024-12-12 09:00:00 | BUY  | 0.099683 BTC @ $100,814.07 | Fee: $10.06

✅ ALL OPERATIONS COMPLETED SUCCESSFULLY!


In [237]:
# CELL 16: COMPARISON WITH LOGISTIC REGRESSION

print("\n" + "=" * 70)
print(" " * 15 + "MODEL COMPARISON")
print(" " * 10 + "XGBoost vs Logistic Regression")
print("=" * 70)

# Try to load logistic regression metrics
try:
    with open(str(MODELS_DIR / 'metrics.json'), 'r') as f:
        lr_metrics = json.load(f)
    
    print("\n📊 VALIDATION SET COMPARISON:")
    print("-" * 70)
    print(f"{'Metric':<20} {'XGBoost':>15} {'Log Regression':>15} {'Improvement':>15}")
    print("-" * 70)
    
    # Validation metrics
    xgb_val_acc = val_accuracy
    lr_val_acc = lr_metrics['validation']['accuracy']
    print(f"{'Accuracy':<20} {xgb_val_acc:>14.4f} {lr_val_acc:>15.4f} {(xgb_val_acc-lr_val_acc)*100:>14.2f}%")
    
    xgb_val_prec = val_precision
    lr_val_prec = lr_metrics['validation']['precision']
    print(f"{'Precision':<20} {xgb_val_prec:>14.4f} {lr_val_prec:>15.4f} {(xgb_val_prec-lr_val_prec)*100:>14.2f}%")
    
    xgb_val_rec = val_recall
    lr_val_rec = lr_metrics['validation']['recall']
    print(f"{'Recall':<20} {xgb_val_rec:>14.4f} {lr_val_rec:>15.4f} {(xgb_val_rec-lr_val_rec)*100:>14.2f}%")
    
    if not np.isnan(val_roc_auc) and lr_metrics['validation']['roc_auc'] is not None:
        xgb_val_auc = val_roc_auc
        lr_val_auc = lr_metrics['validation']['roc_auc']
        print(f"{'ROC-AUC':<20} {xgb_val_auc:>14.4f} {lr_val_auc:>15.4f} {(xgb_val_auc-lr_val_auc)*100:>14.2f}%")
    
    print("\n💰 VALIDATION BACKTEST COMPARISON:")
    print("-" * 70)
    
    xgb_val_return = val_backtest['total_return']
    lr_val_return = lr_metrics['validation']['backtest_return']
    print(f"{'Total Return':<20} {xgb_val_return:>14.2f}% {lr_val_return:>14.2f}% {xgb_val_return-lr_val_return:>14.2f}%")
    
    xgb_val_sharpe = val_backtest['sharpe_ratio']
    lr_val_sharpe = lr_metrics['validation']['sharpe_ratio']
    print(f"{'Sharpe Ratio':<20} {xgb_val_sharpe:>14.2f} {lr_val_sharpe:>15.2f} {xgb_val_sharpe-lr_val_sharpe:>14.2f}")
    
    xgb_val_dd = val_backtest['max_drawdown']
    lr_val_dd = lr_metrics['validation']['max_drawdown']
    print(f"{'Max Drawdown':<20} {xgb_val_dd:>14.2f}% {lr_val_dd:>14.2f}% {xgb_val_dd-lr_val_dd:>14.2f}%")
    
    xgb_val_trades = val_backtest['num_trades']
    lr_val_trades = lr_metrics['validation']['num_trades']
    print(f"{'Number of Trades':<20} {xgb_val_trades:>14} {lr_val_trades:>15} {xgb_val_trades-lr_val_trades:>14}")
    
    print("\n" + "=" * 70)
    print("\n📊 TEST SET COMPARISON:")
    print("-" * 70)
    print(f"{'Metric':<20} {'XGBoost':>15} {'Log Regression':>15} {'Improvement':>15}")
    print("-" * 70)
    
    # Test metrics
    xgb_test_acc = test_accuracy
    lr_test_acc = lr_metrics['test']['accuracy']
    print(f"{'Accuracy':<20} {xgb_test_acc:>14.4f} {lr_test_acc:>15.4f} {(xgb_test_acc-lr_test_acc)*100:>14.2f}%")
    
    xgb_test_return = test_backtest['total_return']
    lr_test_return = lr_metrics['test']['backtest_return']
    print(f"{'Total Return':<20} {xgb_test_return:>14.2f}% {lr_test_return:>14.2f}% {xgb_test_return-lr_test_return:>14.2f}%")
    
    print("\n" + "=" * 70)
    
    # Summary
    print("\n🎯 SUMMARY:")
    if xgb_val_return > lr_val_return and xgb_test_return > lr_test_return:
        print("  ✅ XGBoost OUTPERFORMS Logistic Regression on both validation and test sets")
    elif xgb_val_return > lr_val_return:
        print("  ⚡ XGBoost performs better on validation but mixed results on test set")
    else:
        print("  ⚠️ Results are comparable or need further optimization")
    
except FileNotFoundError:
    print(f"\n⚠️ Logistic Regression metrics not found ({MODELS_DIR / 'metrics.json'})")
    print("   Run the Logistic Regression notebook first to compare models")
except Exception as e:
    print(f"\n❌ Error loading comparison data: {e}")


               MODEL COMPARISON
          XGBoost vs Logistic Regression

📊 VALIDATION SET COMPARISON:
----------------------------------------------------------------------
Metric                       XGBoost  Log Regression     Improvement
----------------------------------------------------------------------
Accuracy                     0.5320          0.5286           0.34%
Precision                    0.5423          0.5460          -0.37%
Recall                       0.5218          0.4439           7.78%
ROC-AUC                      0.5447          0.5432           0.15%

💰 VALIDATION BACKTEST COMPARISON:
----------------------------------------------------------------------
Total Return                   0.00%         -70.79%          70.79%
Sharpe Ratio                   0.00           -3.22           3.22
Max Drawdown                   0.00%         -71.14%          71.14%
Number of Trades                  0            1555          -1555


📊 TEST SET COMPARISON:
----------

In [238]:
# CELL 17: PREDICTION CONFIDENCE ANALYSIS

print("\n" + "=" * 70)
print(" " * 15 + "PREDICTION CONFIDENCE ANALYSIS")
print("=" * 70)

# Analyze prediction confidence distribution
confidence_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
val_confidence = [prob if pred == 1 else 1-prob 
                  for pred, prob in zip(y_val_pred, y_val_proba)]

print("\n📊 Validation Set - Confidence Distribution:")
print("-" * 70)
print(f"{'Confidence Range':<25} {'Count':>15} {'Percentage':>15}")
print("-" * 70)

for i in range(len(confidence_bins)-1):
    lower = confidence_bins[i]
    upper = confidence_bins[i+1]
    count = sum(1 for c in val_confidence if lower <= c < upper)
    pct = count / len(val_confidence) * 100
    print(f"{lower:.1f} - {upper:.1f} {' ':<14} {count:>15,} {pct:>14.1f}%")

print("-" * 70)

# Analyze accuracy by confidence level
print("\n🎯 Accuracy by Confidence Level:")
print("-" * 70)
print(f"{'Confidence Range':<25} {'Accuracy':>15} {'Samples':>15}")
print("-" * 70)

for i in range(len(confidence_bins)-1):
    lower = confidence_bins[i]
    upper = confidence_bins[i+1]
    
    # Get predictions in this confidence range
    mask = [(lower <= c < upper) for c in val_confidence]
    if sum(mask) > 0:
        y_true_subset = y_val.values[mask]
        y_pred_subset = y_val_pred[mask]
        acc = accuracy_score(y_true_subset, y_pred_subset)
        print(f"{lower:.1f} - {upper:.1f} {' ':<14} {acc:>14.4f} {sum(mask):>15,}")
    else:
        print(f"{lower:.1f} - {upper:.1f} {' ':<14} {'N/A':>14} {0:>15}")

print("-" * 70)

# High confidence predictions
high_confidence_threshold = 0.7
high_conf_mask = [c >= high_confidence_threshold for c in val_confidence]
high_conf_count = sum(high_conf_mask)
high_conf_pct = high_conf_count / len(val_confidence) * 100

print(f"\n💡 High Confidence Predictions (≥{high_confidence_threshold}):")
print(f"  • Count: {high_conf_count:,} ({high_conf_pct:.1f}% of total)")

if high_conf_count > 0:
    y_true_high_conf = y_val.values[high_conf_mask]
    y_pred_high_conf = y_val_pred[high_conf_mask]
    high_conf_acc = accuracy_score(y_true_high_conf, y_pred_high_conf)
    print(f"  • Accuracy: {high_conf_acc:.4f}")
    
    improvement = (high_conf_acc - val_accuracy) * 100
    if improvement > 0:
        print(f"  • Improvement over baseline: +{improvement:.2f}%")
        print(f"  ✅ High confidence predictions are MORE accurate")
    else:
        print(f"  • Improvement over baseline: {improvement:.2f}%")
        print(f"  ⚠️ High confidence predictions are NOT more accurate")

print("\n" + "=" * 70)


               PREDICTION CONFIDENCE ANALYSIS

📊 Validation Set - Confidence Distribution:
----------------------------------------------------------------------
Confidence Range                    Count      Percentage
----------------------------------------------------------------------
0.5 - 0.6                          7,715          100.0%
0.6 - 0.7                              1            0.0%
0.7 - 0.8                              0            0.0%
0.8 - 0.9                              0            0.0%
0.9 - 1.0                              0            0.0%
----------------------------------------------------------------------

🎯 Accuracy by Confidence Level:
----------------------------------------------------------------------
Confidence Range                 Accuracy         Samples
----------------------------------------------------------------------
0.5 - 0.6                        0.5320           7,715
0.6 - 0.7                        1.0000               1
0.7 - 0